<a href="https://colab.research.google.com/github/safrinfathima26/Deep_Learning/blob/main/RNN_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#Step 1: Install and Load the Dataset

In [2]:
# STEP 1 : Install and Load the Dataset
# Install required libraries (Run only once)
# !pip install datasets tensorflow scikit-learn
import re
import numpy as np

from datasets import load_dataset
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


In [3]:
# Load the AG News dataset from Hugging Face
dataset = load_dataset("wangrongsheng/ag_news")

# Extract training text and labels
train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]

# Extract testing text and labels
test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

print("Training Samples :", len(train_texts))
print("Testing Samples  :", len(test_texts))

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Training Samples : 120000
Testing Samples  : 7600


In [ ]:
#Step 2: Clean and Normalize the Text

In [4]:
# STEP 2 : Clean and Normalize the Text
# Function to clean text
def clean_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove everything except letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply cleaning to all training texts
train_texts = [clean_text(text) for text in train_texts]

# Apply cleaning to all testing texts
test_texts = [clean_text(text) for text in test_texts]


In [ ]:
#Step 3: Tokenize the Strings into Numbers

In [5]:
# STEP 3 : Tokenize the Strings into Numbers
# Maximum vocabulary size
VOCAB_SIZE = 20000

# Create tokenizer
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")

# Build vocabulary using training data
tokenizer.fit_on_texts(train_texts)

# Convert words into integer sequences
X_train = tokenizer.texts_to_sequences(train_texts)
X_test = tokenizer.texts_to_sequences(test_texts)

In [ ]:
#Step 4: Pad and Truncate Sequences


In [6]:
#STEP 4 : Pad and Truncate Sequences
# Maximum sentence length
MAX_LENGTH = 50

# Pad training sequences
X_train = pad_sequences(
    X_train,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)

# Pad testing sequences
X_test = pad_sequences(
    X_test,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)



In [ ]:
#Step 5: Convert Labels to Categorical Vectors


In [7]:
# STEP 5 : Convert Labels to Categorical Vectors
# AG News has 4 classes
NUM_CLASSES = 4

# Convert labels into one-hot vectors
y_train = to_categorical(train_labels, num_classes=NUM_CLASSES)
y_test = to_categorical(test_labels, num_classes=NUM_CLASSES)



In [ ]:
#Step 6: Define the RNN Model Architecture


In [8]:
# STEP 6 : Define the RNN Model Architecture
# Build Sequential model
model = Sequential()
# Embedding Layer
# Converts word IDs into dense vectors
model.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LENGTH
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
# Simple RNN Layer
# Learns sequential information
# ----------------------------------------------------------
model.add(
    SimpleRNN(
        units=64,
        activation='tanh'
    )
)

In [10]:
# Output Layer
# Softmax gives probability for each class
# ----------------------------------------------------------
model.add(
    Dense(
        NUM_CLASSES,
        activation='softmax'
    )
)

# Display model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Step 7: Compile and Train the Model

In [11]:
# STEP 7 : Compile and Train the Model
# Compile the model
model.compile(

    optimizer='adam',

    loss='categorical_crossentropy',

    metrics=['accuracy']
)

# Train the model
history = model.fit(

    X_train,

    y_train,

    epochs=5,

    batch_size=64,

    validation_split=0.2

)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 42s 27ms/step - accuracy: 0.7482 - loss: 0.7151 - val_accuracy: 0.7802 - val_loss: 0.6211
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 42s 28ms/step - accuracy: 0.8652 - loss: 0.4278 - val_accuracy: 0.8414 - val_loss: 0.4831
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 39s 26ms/step - accuracy: 0.8941 - loss: 0.3398 - val_accuracy: 0.8460 - val_loss: 0.4532
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - accuracy: 0.9005 - loss: 0.3205 - val_accuracy: 0.8349 - val_loss: 0.4912
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 39s 26ms/step - accuracy: 0.8106 - loss: 0.4918 - val_accuracy: 0.4061 - val_loss: 1.2026


In [ ]:
#Step 8: Read and Evaluate the Target Accuracy


In [14]:
# STEP 8 : Read and Evaluate the Target Accuracy
# Evaluate model using test data
loss, accuracy = model.evaluate(X_test, y_test)

# Print final results
print("Test Loss     :", loss)
print("Test Accuracy :", accuracy)

# Predict a sample sentence
sample = "Apple releases a new AI powered iPhone."

# Clean text
sample = clean_text(sample)

# Convert to sequence
sample_seq = tokenizer.texts_to_sequences([sample])

# Pad sequence
sample_pad = pad_sequences(
    sample_seq,
    maxlen=MAX_LENGTH,
    padding='post'
)

# Predict class
prediction = model.predict(sample_pad)

predicted_class = np.argmax(prediction)

# AG News labels
labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Science/Technology"
}

print("\nPredicted Category :", labels[predicted_class])

238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4064 - loss: 1.1823
Test Loss     : 1.1822917461395264
Test Accuracy : 0.4064473807811737
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

Predicted Category : Business
